[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C47_RecSys_Ranking_Course/01_collaborative_filtering/01_collaborative_filtering.ipynb)

# 01 · 协同过滤（用 numpy 从零）

目标：把 **item-based / user-based 协同过滤** 从零实现——**cosine/Pearson/调整余弦相似度**、**top-k 邻域加权预测**、**隐式反馈共现相似度**、**Top-N 评估**，全部对拍验证。

路线：复用模块 00 数据 → 相似度三件套 → item-based 预测 → user-based 对比 → 隐式共现 → ✏️ 练习 → 📖 答案 → 🧪 真实 MovieLens 评估胶囊。

> 核心心智：**CF = 用相似度加权已知评分来填空/排序，相似度从交互模式里来，不看物品内容。**

## 0 · 复用模块 00 的数据基座

把模块 00 的加载器复制过来（每个 notebook 自包含，可独立运行）。真实 MovieLens 优先，失败回退合成。

In [ ]:
import numpy as np

def load_movielens_or_synth(n_users=200, n_items=300, rank=8, seed=0, verbose=True):
    import os
    for path in ['ml-100k/u.data', 'u.data', os.path.expanduser('~/ml-100k/u.data')]:
        if os.path.exists(path):
            data = np.loadtxt(path, dtype=np.int64)[:, :3].astype(float)
            data[:, 0] -= 1; data[:, 1] -= 1
            nu = int(data[:, 0].max()) + 1; ni = int(data[:, 1].max()) + 1
            if verbose: print(f'真实 MovieLens-100k: {len(data)} 评分')
            return data, nu, ni
    try:
        import urllib.request
        url = 'https://files.grouplens.org/datasets/movielens/ml-100k/u.data'
        with urllib.request.urlopen(url, timeout=5) as r:
            raw = r.read().decode()
        rows = [list(map(int, ln.split('\t')[:3])) for ln in raw.strip().split('\n')]
        data = np.array(rows, dtype=float); data[:, 0] -= 1; data[:, 1] -= 1
        return data, int(data[:,0].max())+1, int(data[:,1].max())+1
    except Exception as e:
        if verbose: print(f'回退合成数据（{type(e).__name__}）')
    rng = np.random.default_rng(seed)
    P = rng.standard_normal((n_users, rank)) * 0.5
    Q = rng.standard_normal((n_items, rank)) * 0.5
    bu = rng.standard_normal(n_users) * 0.3; bi = rng.standard_normal(n_items) * 0.5
    rows = []
    for u in range(n_users):
        items = rng.choice(n_items, size=rng.integers(20, 60), replace=False)
        for i in items:
            r = 3.5 + bu[u] + bi[i] + P[u] @ Q[i] + rng.standard_normal()*0.3
            rows.append([u, i, float(np.clip(np.round(r*2)/2, 1, 5))])
    data = np.array(rows, dtype=float)
    if verbose: print(f'合成评分: {len(data)} 条 (seed={seed})')
    return data, n_users, n_items

def to_dense_matrix(ratings, n_users, n_items):
    R = np.zeros((n_users, n_items))
    R[ratings[:,0].astype(int), ratings[:,1].astype(int)] = ratings[:,2]
    return R

ratings, n_users, n_items = load_movielens_or_synth(seed=0)
R = to_dense_matrix(ratings, n_users, n_items)
print('R 形状', R.shape, '| 稀疏度', round(1 - (R>0).sum()/R.size, 3))
assert R.shape == (n_users, n_items)
print('✅ 数据就绪')

## 1 · 物品余弦相似度（向量化）

把每个物品看成它在用户空间的评分列向量 $r_{\cdot i}$，两物品相似度 = 夹角余弦。
向量化技巧：先按列归一化（每列除以其 L2 范数），相似度矩阵就是 $\hat R^\top \hat R$。

我们写一个向量化版，再写一个**朴素逐对循环**版对拍，确保向量化没写错。

In [ ]:
def item_cosine_sim(R):
    '''物品-物品余弦相似度矩阵 (n_items, n_items)。未评为 0。'''
    norms = np.linalg.norm(R, axis=0)               # 每个物品列的范数
    norms_safe = np.where(norms == 0, 1.0, norms)    # 防除零
    Rn = R / norms_safe                              # 列归一化
    S = Rn.T @ Rn                                    # 归一化后内积 = 余弦
    return S

def item_cosine_sim_naive(R):
    n = R.shape[1]
    S = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            a, b = R[:, i], R[:, j]
            na, nb = np.linalg.norm(a), np.linalg.norm(b)
            S[i, j] = (a @ b) / (na * nb) if na > 0 and nb > 0 else 0.0
    return S

# 小子集对拍（朴素版 O(n^2) 慢，取前 40 个物品）
Rsub = R[:, :40]
S_fast = item_cosine_sim(Rsub)
S_slow = item_cosine_sim_naive(Rsub)
assert np.allclose(S_fast, S_slow, atol=1e-10), '向量化与朴素版应一致'
assert np.allclose(np.diag(S_fast), 1.0, atol=1e-9) or (np.diag(S_fast) <= 1.0+1e-9).all()
print('对拍通过 | 自相似(对角) 应≈1:', round(float(np.diag(S_fast)[0]), 4))
print('余弦范围:', round(float(S_fast.min()), 3), '~', round(float(S_fast.max()), 3))
print('✅ 向量化余弦 == 朴素逐对余弦')

## 2 · 调整余弦（减用户均值）—— item-based 的推荐选择

裸余弦没扣除「用户个人打分尺度」（有人爱打高分）。**调整余弦**先减去每个**用户**的均值再算物品间余弦（Sarwar 2001 证明它在 item-based 上更优）。

关键：均值只在**已评**的项上算，且中心化也只作用于已评项（未评仍是 0，不参与）。

In [ ]:
def adjusted_cosine_sim(R):
    '''调整余弦：减去每个用户在其已评物品上的均值，再算物品间余弦。'''
    mask = (R > 0).astype(float)
    user_sum = R.sum(axis=1)
    user_cnt = mask.sum(axis=1)
    user_mean = np.where(user_cnt > 0, user_sum / np.where(user_cnt==0,1,user_cnt), 0.0)
    Rc = (R - user_mean[:, None]) * mask            # 只对已评项中心化，未评保持 0
    norms = np.linalg.norm(Rc, axis=0)
    norms_safe = np.where(norms == 0, 1.0, norms)
    Rn = Rc / norms_safe
    return Rn.T @ Rn

S_adj = adjusted_cosine_sim(R)
print('调整余弦范围:', round(float(S_adj.min()), 3), '~', round(float(S_adj.max()), 3))
# 调整余弦会出现负相似度（品味相反），裸余弦(全正评分)几乎全正
assert S_adj.min() < 0, '中心化后应出现负相似度（品味相反的物品）'
assert S_adj.shape == (n_items, n_items)
print('✅ 调整余弦出现负值——这是中心化的正确效果（裸余弦在全正评分下几乎全正，区分度差）')

## 3 · item-based 预测：top-k 邻域加权

预测用户 $u$ 对物品 $i$：用 $u$ **评过**的、与 $i$ 最相似的 $k$ 个**正相似度**物品，按相似度加权平均：
$$\hat r_{ui} = \frac{\sum_{j\in N_k^+(i;u)} s_{ij}\, r_{uj}}{\sum_{j\in N_k^+(i;u)} s_{ij}}$$

**只取正相似度邻居**是关键：调整余弦会产生负相似度（品味相反），把它们也加进来再用 `|s|` 归一会得到没有意义的预测（负邻居把评分往乱拉）。经典 item-based CF（Sarwar 2001）正是用正相似邻居加权——这能稳定地优于基线。

In [ ]:
def predict_itemcf(u, i, R, S, k=20):
    '''item-based CF 预测单个 (u,i) 评分（top-k 正相似度邻居加权）。'''
    rated = np.where(R[u] > 0)[0]                   # u 评过的物品
    rated = rated[rated != i]                        # 排除 i 自己
    if len(rated) == 0:
        return R[R>0].mean()                         # 退化：返回全局均值
    sims = S[i, rated]                               # i 与这些物品的相似度
    pos = sims > 0                                   # 只用正相似度邻居（关键！）
    rated, sims = rated[pos], sims[pos]
    if len(rated) == 0:
        return R[u][R[u] > 0].mean()                 # 无正邻居：退化为该用户均值
    topk = np.argsort(-sims)[:k]                    # 按相似度取 top-k
    nb_items = rated[topk]; nb_sims = sims[topk]
    denom = nb_sims.sum()
    if denom < 1e-12:
        return R[u, rated].mean()
    return float((nb_sims * R[u, nb_items]).sum() / denom)

S = adjusted_cosine_sim(R)
# 找一个有评分的 (u,i)，遮住它、预测、对比真值
u0, i0 = int(ratings[0,0]), int(ratings[0,1])
true_r = R[u0, i0]
R_masked = R.copy(); R_masked[u0, i0] = 0           # 遮住这一格
S_masked = adjusted_cosine_sim(R_masked)
pred = predict_itemcf(u0, i0, R_masked, S_masked, k=20)
print(f'用户{u0} 物品{i0}: 真实评分={true_r}, CF预测={pred:.3f}')
assert 1.0 <= pred <= 5.0, '预测应在合理评分范围'
print('✅ item-based 预测可运行且落在合理区间')

### 评估：item-based CF 的 RMSE（vs 全局均值基线）

在一批遮住的评分上测 RMSE，和「永远预测全局均值」的基线比——CF 应该明显更好。

In [ ]:
def rmse_itemcf(ratings, n_users, n_items, k=20, n_test=300, seed=1):
    rng = np.random.default_rng(seed)
    idx = rng.choice(len(ratings), size=min(n_test, len(ratings)), replace=False)
    R_full = to_dense_matrix(ratings, n_users, n_items)
    R_tr = R_full.copy()
    for t in idx:
        u, i = int(ratings[t,0]), int(ratings[t,1])
        R_tr[u, i] = 0                              # 从训练里遮掉测试项
    S = adjusted_cosine_sim(R_tr)
    global_mean = R_tr[R_tr>0].mean()
    se_cf, se_base = 0.0, 0.0
    for t in idx:
        u, i = int(ratings[t,0]), int(ratings[t,1])
        true = ratings[t,2]
        se_cf += (predict_itemcf(u, i, R_tr, S, k) - true)**2
        se_base += (global_mean - true)**2
    return (se_cf/len(idx))**0.5, (se_base/len(idx))**0.5

rmse_cf, rmse_base = rmse_itemcf(ratings, n_users, n_items, k=20)
print(f'item-based CF RMSE = {rmse_cf:.4f}')
print(f'全局均值基线 RMSE  = {rmse_base:.4f}')
assert rmse_cf < rmse_base, 'CF 应优于全局均值基线'
print('✅ CF 明显优于「永远猜平均分」的基线——协同信号确实有用')

## 4 · user-based CF：对偶视角

把矩阵转置就把 item-based 变成 user-based——找相似**用户**、用他们的评分预测。
下面直接复用相似度函数（喂 `R.T`）展示对偶性，并对比两者预测。

In [ ]:
def predict_usercf(u, i, R, S_user, k=20):
    '''user-based: 用评过 i 的、与 u 最相似的 k 个正相似用户加权。'''
    raters = np.where(R[:, i] > 0)[0]              # 评过 i 的用户
    raters = raters[raters != u]
    if len(raters) == 0:
        return R[R>0].mean()
    sims = S_user[u, raters]
    pos = sims > 0; raters, sims = raters[pos], sims[pos]   # 只用正相似邻居
    if len(raters) == 0:
        return R[R>0].mean()
    topk = np.argsort(-sims)[:k]
    nb = raters[topk]; ns = sims[topk]
    denom = ns.sum()
    return float((ns * R[nb, i]).sum()/denom) if denom>1e-12 else R[raters, i].mean()

# user 相似度 = 对 R.T 算调整余弦（这里减物品均值，等价于在用户空间的中心化余弦）
S_user = item_cosine_sim(R.T)        # 用户间余弦（行=用户）
u0, i0 = int(ratings[5,0]), int(ratings[5,1])
R_m = R.copy(); R_m[u0,i0] = 0
S_user_m = item_cosine_sim(R_m.T)
S_item_m = adjusted_cosine_sim(R_m)
p_user = predict_usercf(u0, i0, R_m, S_user_m, k=20)
p_item = predict_itemcf(u0, i0, R_m, S_item_m, k=20)
print(f'真值={R[u0,i0]} | user-based={p_user:.3f} | item-based={p_item:.3f}')
assert 1.0 <= p_user <= 5.0 and 1.0 <= p_item <= 5.0
print('✅ 两个对偶视角都能预测；工业上 item-based 更常用（物品相似度更稳定、可离线预算）')

## 5 · 隐式反馈：共现相似度（Amazon「买了又买」）

隐式反馈下把矩阵二值化（交互=1）。两物品余弦相似度 = $\frac{|U_i\cap U_j|}{\sqrt{|U_i|}\sqrt{|U_j|}}$——
**共同用户数，按各自流行度归一**。分母归一是为了**压制流行度偏置**（不然爆款和谁都像）。

In [ ]:
def implicit_cooccur_sim(R, threshold=4.0):
    '''二值化(>=threshold 算正向交互)后的物品共现余弦相似度。'''
    B = (R >= threshold).astype(float)             # 二值交互矩阵
    cooccur = B.T @ B                               # cooccur[i,j] = |U_i ∩ U_j|
    pop = np.sqrt(np.diag(cooccur))                 # sqrt(|U_i|)
    pop_safe = np.where(pop == 0, 1.0, pop)
    S = cooccur / np.outer(pop_safe, pop_safe)      # 流行度归一
    return S, B

S_imp, B = implicit_cooccur_sim(R, threshold=4.0)
# 验证：对角(自共现归一)应=1（对有交互的物品）；归一化压制了流行度
active = np.where(B.sum(axis=0) > 0)[0]
assert np.allclose(np.diag(S_imp)[active], 1.0, atol=1e-9)
# 对比：不归一化时，最热门物品与所有东西的共现都最大（流行度偏置）
raw_cooccur = (B.T @ B)
most_pop = int(B.sum(axis=0).argmax())
print(f'最热门物品 {most_pop} 被 {int(B.sum(axis=0)[most_pop])} 人交互')
print(f'不归一化: 它与其他物品的平均共现 = {raw_cooccur[most_pop].mean():.2f}（虚高，因为它太热门）')
print(f'归一化后: 它与其他物品的平均相似度 = {S_imp[most_pop].mean():.3f}（被流行度归一压回）')
assert S_imp[most_pop].mean() < 1.0
print('✅ 归一化把流行度偏置压住了——这是隐式 CF 不推全是爆款的关键')

---
## ✏️ 练习 1：Pearson 相关相似度

实现物品间的 **Pearson 相关**（中心化余弦）：只在**共同评分**的用户上，先减去各物品的均值再算余弦。
$$\text{pearson}(i,j)=\frac{\sum_{u\in U_{ij}}(r_{ui}-\bar r_i)(r_{uj}-\bar r_j)}{\sqrt{\sum(r_{ui}-\bar r_i)^2}\sqrt{\sum(r_{uj}-\bar r_j)^2}}$$
为简化，对一对物品 `(i,j)` 实现即可（不必向量化全矩阵）。

In [ ]:
def pearson_pair(R, i, j):
    '''物品 i,j 的 Pearson 相关，只在同时评过两者的用户上算。无共同评分返回 0。'''
    # TODO:
    #  1) co = 同时评过 i 和 j 的用户下标 (R[:,i]>0) & (R[:,j]>0)
    #  2) 若 co 少于 2 个，返回 0.0
    #  3) ri, rj = R[co,i], R[co,j]; 各自减去在 co 上的均值
    #  4) 返回中心化向量的余弦；分母为 0 返回 0.0
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
# 构造已知答案的小例子
Rt = np.array([
    [5., 4., 0.],
    [3., 2., 0.],
    [4., 3., 1.],
])
# 物品0 与 物品1: 共同用户=全部3个, 中心化后完全正相关 -> +1
p01 = pearson_pair(Rt, 0, 1)
assert abs(p01 - 1.0) < 1e-9, f'完全正相关应为1, 得到 {p01}'
# 物品0 与 物品2: 只有用户2同时评过(1个共同) -> 0
assert pearson_pair(Rt, 0, 2) == 0.0, '共同评分<2 应返回0'
# 反相关检验
Rneg = np.array([[5.,1.],[4.,2.],[3.,3.],[2.,4.],[1.,5.]])
assert abs(pearson_pair(Rneg, 0, 1) + 1.0) < 1e-9, '完全负相关应为 -1'
print('✅ 练习 1 通过：Pearson 相关正确（含正相关/负相关/共评不足）')

## ✏️ 练习 2：相似度收缩（shrinkage）

共同评分用户太少时相似度不可靠。实现**收缩**：把相似度乘以 $\frac{n_{ij}}{n_{ij}+\lambda}$，
其中 $n_{ij}$ 是共同评分用户数。共评越少，越往 0 压。

In [ ]:
def shrink_similarity(sim, n_common, lam=10.0):
    '''对相似度做收缩：sim * n/(n+lam)。'''
    # TODO: 返回收缩后的相似度
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
# 共评很多 -> 几乎不收缩；共评很少 -> 强烈收缩
assert abs(shrink_similarity(0.9, n_common=1000, lam=10) - 0.9*1000/1010) < 1e-9
assert shrink_similarity(0.9, n_common=1, lam=10) < 0.1, '共评=1 应被压到很小'
assert shrink_similarity(0.9, n_common=10, lam=10) == 0.9*0.5, 'n=lam 时收缩到一半'
# 单调性：共评越多收缩越弱
vals = [shrink_similarity(0.8, n, 10) for n in [1, 5, 50, 500]]
assert all(vals[i] < vals[i+1] for i in range(len(vals)-1)), '应随共评数单调上升'
print('✅ 练习 2 通过：收缩把「共评不足的高相似度」正确压低')

## ✏️ 练习 3：Top-N 推荐 + Recall@K 评估

把 item-based CF 用于**生成 Top-N 列表**并评估。实现 `recommend_topn`：
对用户 $u$ **所有未评物品**算 CF 预测分，返回分最高的 N 个物品 id（降序）。

In [ ]:
def recommend_topn(u, R, S, k=20, n=10):
    '''对用户 u 的所有未评物品打 CF 分，返回 Top-N 物品 id（降序）。'''
    # TODO:
    #  1) unrated = u 未评过的物品下标 (R[u]==0)
    #  2) 对每个 unrated 物品算 predict_itemcf(u, i, R, S, k)
    #  3) 返回分数最高的 n 个物品 id（按分数降序）
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
def recall_at_k(ranked, relevant_set, k):
    if not relevant_set: return 0.0
    return sum(1 for it in ranked[:k] if it in relevant_set)/len(relevant_set)

# leave-one-out：给若干用户遮掉一个高分物品，看 CF 能否把它排进 Top-N
rng = np.random.default_rng(2)
hits = 0; tries = 0
for u in rng.choice(n_users, size=30, replace=False):
    liked = np.where(R[u] >= 4.0)[0]
    if len(liked) < 2: continue
    held = int(rng.choice(liked))
    R_m = R.copy(); R_m[u, held] = 0
    S_m = adjusted_cosine_sim(R_m)
    topn = recommend_topn(u, R_m, S_m, k=20, n=20)
    assert len(topn) == 20 and held not in np.where(R_m[u]>0)[0], 'Top-N 不应含已评物品'
    hits += recall_at_k(topn, {held}, 20); tries += 1
recall = hits/tries
print(f'Recall@20 (leave-one-out, {tries} 用户) = {recall:.3f}')
# 随机基线命中率 ~ 20/n_items；CF 应明显更高
random_baseline = 20/n_items
print(f'随机基线 ≈ {random_baseline:.3f}')
assert recall > random_baseline, 'CF 的 Recall 应高于随机'
print('✅ 练习 3 通过：CF Top-N 推荐的 Recall 明显高于随机基线')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def pearson_pair(R, i, j):
    co = np.where((R[:, i] > 0) & (R[:, j] > 0))[0]
    if len(co) < 2:
        return 0.0
    ri = R[co, i] - R[co, i].mean()
    rj = R[co, j] - R[co, j].mean()
    denom = np.linalg.norm(ri) * np.linalg.norm(rj)
    return float(ri @ rj / denom) if denom > 1e-12 else 0.0

# 练习 2 参考答案
def shrink_similarity(sim, n_common, lam=10.0):
    return sim * n_common / (n_common + lam)

# 练习 3 参考答案
def recommend_topn(u, R, S, k=20, n=10):
    unrated = np.where(R[u] == 0)[0]
    scores = np.array([predict_itemcf(u, int(i), R, S, k) for i in unrated])
    order = np.argsort(-scores)[:n]
    return [int(unrated[o]) for o in order]
print('参考答案已载入')

---
## 🧪 真实数据胶囊：item-based CF 在 MovieLens 上找「相似电影」

协同过滤最可解释的产物是「**相似物品**」——「喜欢这部的人也喜欢…」。
在真实/合成 MovieLens 上，对一个热门电影列出它的 top 相似电影，并验证「相似度对称、自相似最高」这些必须成立的性质。

**TODO**：补全 `top_similar_items`，返回与给定物品最相似的 `topn` 个物品（排除自己）。

In [ ]:
def top_similar_items(S, item_id, topn=5):
    '''返回与 item_id 最相似的 topn 个物品 (id, 相似度)，排除自己。'''
    sims = S[item_id].copy()
    sims[item_id] = -np.inf                          # 排除自己
    # TODO: 取相似度最高的 topn 个，返回 [(item_id, sim), ...]
    raise NotImplementedError

In [ ]:
# 自测（胶囊）
S_imp, B = implicit_cooccur_sim(R, threshold=4.0)
hot = int(B.sum(axis=0).argmax())                    # 最热门电影
neighbors = top_similar_items(S_imp, hot, topn=5)
print(f'与热门电影 {hot} 最相似的 5 部:')
for iid, sim in neighbors:
    print(f'   电影 {iid}: 相似度 {sim:.3f}')
assert len(neighbors) == 5
assert all(iid != hot for iid, _ in neighbors), '不能包含自己'
assert neighbors[0][1] >= neighbors[-1][1], '应按相似度降序'
# 相似度矩阵必须对称
assert np.allclose(S_imp, S_imp.T, atol=1e-9), '相似度矩阵应对称'
print('✅ 胶囊通过：相似度对称、降序、排除自身——这就是「看了又看」的引擎')

In [ ]:
# 📖 胶囊参考答案
def top_similar_items(S, item_id, topn=5):
    sims = S[item_id].copy()
    sims[item_id] = -np.inf
    order = np.argsort(-sims)[:topn]
    return [(int(o), float(sims[o])) for o in order]

### 小结
- **协同过滤** = 只用交互模式（不看内容）推荐；核心假设：相似的人喜欢相似的物品。
- **相似度三件套**：余弦（默认）、Pearson（中心化）、**调整余弦**（减用户均值，item-based 最优）；共评少要**收缩**。
- **item-based** 比 user-based 更稳定、可离线预算（Amazon「看了又看」）；预测 = **top-k 邻域加权**。
- **隐式反馈**用共现相似度，分母**流行度归一**压制爆款偏置；**冷启动**是纯 CF 的死穴，要靠内容补位。

下一站：**模块 02 · 矩阵分解** —— 不再数邻居，而是把交互矩阵分解成用户/物品的隐因子向量。